In [5]:
# ============================================
# Movie Recommendation System
# Collaborative Filtering | MovieLens Dataset
# ============================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# --- Data Loading ---
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q ml-latest-small.zip

ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

# --- Data Preprocessing ---
df = pd.merge(ratings, movies, on='movieId')
df = df.drop(columns=['timestamp', 'movieId'])

# --- User-Item Matrix ---
user_movie_matrix = df.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)
matrix_filled = user_movie_matrix.fillna(0)

# --- Cosine Similarity ---
user_similarity = cosine_similarity(matrix_filled)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

# --- Recommendation Function ---
def recommend_movies(user_id, num_recommendations=5):
    similar_users = user_similarity_df[user_id].drop(user_id)
    top_users = similar_users.sort_values(ascending=False).head(10)
    watched = user_movie_matrix.loc[user_id]
    watched_movies = watched[watched > 0].index.tolist()
    scores = {}
    for similar_user, similarity in top_users.items():
        ratings_su = user_movie_matrix.loc[similar_user]
        unseen = ratings_su[~ratings_su.index.isin(watched_movies)]
        for movie, rating in unseen.items():
            if rating > 0:
                if movie not in scores:
                    scores[movie] = 0
                scores[movie] += similarity * rating
    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [movie for movie, score in recommended[:num_recommendations]]

# --- Prediction Function ---
def predict_rating(user_id, movie):
    if movie not in user_movie_matrix.columns:
        return 0
    similar_users = user_similarity_df[user_id].drop(user_id)
    top_users = similar_users.sort_values(ascending=False).head(10)
    numerator = 0
    denominator = 0
    for similar_user, similarity in top_users.items():
        rating = user_movie_matrix.loc[similar_user, movie]
        if rating > 0:
            numerator += similarity * rating
            denominator += similarity
    if denominator == 0:
        return 0
    return numerator / denominator

# --- RMSE Evaluation ---
all_ratings = []
for user_id in user_movie_matrix.index:
    watched = user_movie_matrix.loc[user_id]
    watched = watched[watched > 0]
    for movie, rating in watched.items():
        all_ratings.append((user_id, movie, rating))

train_data, test_data = train_test_split(
    all_ratings, test_size=0.2, random_state=42
)

actuals = []
predictions = []
for user_id, movie, actual_rating in test_data[:500]:
    predicted = predict_rating(user_id, movie)
    if predicted > 0:
        actuals.append(actual_rating)
        predictions.append(predicted)

rmse = np.sqrt(mean_squared_error(actuals, predictions))
print(f"RMSE: {rmse:.4f}")

# --- Test Recommendations ---
print("\nUser 1 ke liye recommendations:")
for i, movie in enumerate(recommend_movies(1), 1):
    print(f"{i}. {movie}")

replace ml-latest-small/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
RMSE: 0.9754

User 1 ke liye recommendations:
1. Terminator 2: Judgment Day (1991)
2. Aliens (1986)
3. Sixth Sense, The (1999)
4. Hunt for Red October, The (1990)
5. Godfather, The (1972)
